<a href="https://colab.research.google.com/github/Nour-Tamimi/BinXtraining/blob/main/Week8/Day1/Day1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
!pip install nltk -q

import nltk
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('averaged_perceptron_tagger')
nltk.download('averaged_perceptron_tagger_eng')

import regex as re
import string
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!


In [5]:
# Tokinizing using manual splitting
raw_text = "The movie wasn't as bad as I expected, but it wasn't great either."

def split_words(text):
    return re.findall(r"[\w]+|[.,!?;]", text)

split_words(raw_text)

['The',
 'movie',
 'wasn',
 't',
 'as',
 'bad',
 'as',
 'I',
 'expected',
 ',',
 'but',
 'it',
 'wasn',
 't',
 'great',
 'either',
 '.']

In [6]:
# Tokinizing using word_tokenize
tokens = word_tokenize(raw_text)
print(tokens)

['The', 'movie', 'was', "n't", 'as', 'bad', 'as', 'I', 'expected', ',', 'but', 'it', 'was', "n't", 'great', 'either', '.']


**Why word_tokenize:** It correctly splits contractions into separate
tokens (e.g. "wasn't" -> "was" + "n't"), which lets us explicitly detect
and preserve negation tokens ("not", "n't", "never") during stopword
removal — critical for sentiment tasks where negation flips meaning.

In [8]:
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

def clean_pipeline(text):
    # 1. Lowercase
    text = text.lower()
    # 2. Tokenize
    tokens = word_tokenize(text)
    # 3. Remove punctuation
    tokens = [t for t in tokens if t not in string.punctuation]
    # 4. Remove stop words
    tokens = [t for t in tokens if t not in stop_words]
    # 5. Lemmatize
    tokens = [lemmatizer.lemmatize(t) for t in tokens]
    return tokens


['movie', "n't", 'bad', 'expected', "n't", 'great', 'either']


In [9]:
negation_words = {"not", "no", "nor", "never", "none", "n't"}

# Check which negation-related tokens got removed by your stopword filter
lost_words = [t for t in word_tokenize(raw_text.lower()) if t in stop_words and t in negation_words]
print("Negation words removed by stopword filter:", lost_words)

# Fixed pipeline: keep negations
custom_stop_words = stop_words - negation_words

def clean_pipeline_preserve_negation(text):
    text = text.lower()
    tokens = word_tokenize(text)
    tokens = [t for t in tokens if t not in string.punctuation]
    tokens = [t for t in tokens if t not in custom_stop_words]
    tokens = [lemmatizer.lemmatize(t) for t in tokens]
    return tokens

cleaned_v2 = clean_pipeline_preserve_negation(raw_text)
print(cleaned_v2)

Negation words removed by stopword filter: []
['movie', "n't", 'bad', 'expected', "n't", 'great', 'either']


## Text Cleaning Pipeline — Design Decisions

**Pipeline steps:** lowercase → tokenize → remove punctuation → remove stop words → lemmatize

**Key decision: negation preservation**
Standard NLTK stopword lists include negation terms (not, no, never, n't).
Since this task involves sentiment, removing these would flip meaning
(e.g. "not good" → "good"). We removed negation terms from the stopword
list before filtering, so they survive cleaning.

**Other choices:**
- Lemmatization over stemming: preserves real words for readability
  and avoids over-aggressive stemming (e.g. "better" → "better", not "bet")
- Punctuation removed after tokenization, not before, so tokenizer
  handles contractions and sentence boundaries correctly

In [18]:
# Practicing on a Text dataset
import pandas as pd

# Load a clean, direct-link CSV version of the tiny-shakespeare dataset
from datasets import load_dataset
ds = load_dataset("Trelis/tiny-shakespeare")

# view
print("ORIGINAL DATA SAMPLE")
# Access the 'train' split and then the 'text' column, taking the first elements
print(ds['train']['Text'][:1])


ORIGINAL DATA SAMPLE
["First Citizen:\nBefore we proceed any further, hear me speak.\n\nAll:\nSpeak, speak.\n\nFirst Citizen:\nYou are all resolved rather to die than to famish?\n\nAll:\nResolved. resolved.\n\nFirst Citizen:\nFirst, you know Caius Marcius is chief enemy to the people.\n\nAll:\nWe know't, we know't.\n\nFirst Citizen:\nLet us kill him, and we'll have corn at our own price.\nIs't a verdict?\n\nAll:\nNo more talking on't; let it be done: away, away!\n\nSecond Citizen:\nOne word, good citizens.\n\nFirst Citizen:\nWe are accounted poor citizens, the patricians good.\nWhat authority surfeits on would relieve us: if they\nwould yield us but the superfluity, while it were\nwholesome, we might guess they relieved us humanely;\nbut they think we are too dear: the leanness that\nafflicts us, the object of our misery, is as an\ninventory to particularise their abundance; our\nsufferance is a gain to them Let us revenge this with\nour pikes, ere we become rakes: for the gods know I\

In [22]:
# preprocessing pipline
cleaned_train_dataset = ds['train'].map(lambda example: {'cleaned_text': clean_pipeline(example['Text'])})
cleaned_test_dataset = ds['test'].map(lambda example: {'cleaned_text': clean_pipeline(example['Text'])})


print(cleaned_train_dataset['cleaned_text'][0])

Map:   0%|          | 0/472 [00:00<?, ? examples/s]

Map:   0%|          | 0/49 [00:00<?, ? examples/s]

['first', 'citizen', 'proceed', 'hear', 'speak', 'speak', 'speak', 'first', 'citizen', 'resolved', 'rather', 'die', 'famish', 'resolved', 'resolved', 'first', 'citizen', 'first', 'know', 'caius', 'marcius', 'chief', 'enemy', 'people', "know't", "know't", 'first', 'citizen', 'let', 'u', 'kill', "'ll", 'corn', 'price', "is't", 'verdict', 'talking', "n't", 'let', 'done', 'away', 'away', 'second', 'citizen', 'one', 'word', 'good', 'citizen', 'first', 'citizen', 'accounted', 'poor', 'citizen', 'patrician', 'good', 'authority', 'surfeit', 'would', 'relieve', 'u', 'would', 'yield', 'u', 'superfluity', 'wholesome', 'might', 'guess', 'relieved', 'u', 'humanely', 'think', 'dear', 'leanness', 'afflicts', 'u', 'object', 'misery', 'inventory', 'particularise', 'abundance', 'sufferance', 'gain', 'let', 'u', 'revenge', 'pike', 'ere', 'become', 'rake', 'god', 'know', 'speak', 'hunger', 'bread', 'thirst', 'revenge', 'second', 'citizen', 'would', 'proceed', 'especially', 'caius', 'marcius', 'first', "'s

In [31]:
# Check which negation-related tokens got removed by your stopword filter
lost_words = [t for t in word_tokenize(ds['train']['Text'][0].lower()) if t in stop_words and t in negation_words]
print("Negation words removed by stopword filter:", lost_words)

# Fixed pipeline: keep negations
custom_stop_words = stop_words - negation_words

def clean_pipeline_preserve_negation(text):
    text = text.lower()
    tokens = word_tokenize(text)
    tokens = [t for t in tokens if t not in string.punctuation]
    tokens = [t for t in tokens if t not in custom_stop_words]
    tokens = [lemmatizer.lemmatize(t) for t in tokens]
    return tokens

cleaned_train_dataset_v2 = ds['train'].map(lambda example: {'cleaned_text': clean_pipeline_preserve_negation(example['Text'])})
cleaned_test_dataset_v2 = ds['test'].map(lambda example: {'cleaned_text': clean_pipeline_preserve_negation(example['Text'])})

print(cleaned_train_dataset_v2["cleaned_text"][0])

Negation words removed by stopword filter: ['no', 'not', 'not', 'not', 'no', 'not', 'not', 'not', 'not', 'not', 'not']
['first', 'citizen', 'proceed', 'hear', 'speak', 'speak', 'speak', 'first', 'citizen', 'resolved', 'rather', 'die', 'famish', 'resolved', 'resolved', 'first', 'citizen', 'first', 'know', 'caius', 'marcius', 'chief', 'enemy', 'people', "know't", "know't", 'first', 'citizen', 'let', 'u', 'kill', "'ll", 'corn', 'price', "is't", 'verdict', 'no', 'talking', "n't", 'let', 'done', 'away', 'away', 'second', 'citizen', 'one', 'word', 'good', 'citizen', 'first', 'citizen', 'accounted', 'poor', 'citizen', 'patrician', 'good', 'authority', 'surfeit', 'would', 'relieve', 'u', 'would', 'yield', 'u', 'superfluity', 'wholesome', 'might', 'guess', 'relieved', 'u', 'humanely', 'think', 'dear', 'leanness', 'afflicts', 'u', 'object', 'misery', 'inventory', 'particularise', 'abundance', 'sufferance', 'gain', 'let', 'u', 'revenge', 'pike', 'ere', 'become', 'rake', 'god', 'know', 'speak', 'h